# colab_07 — Stratified subsampling to 100k per dataset

## Motivation

Session 15 showed DPT fails on the integrated Harmony object. Root causes: (1) ca. 100× cell imbalance between Bhaduri 2020 organoids (ca. 242k) and Zhong 2018 fetal (ca. 2.4k); (2) disconnected graph components; (3) stale `iroot` propagation through `.copy()`.

Session 16 pivoted: replace Zhong with **Bhaduri 2021** (atlas of cortical arealization; ca. 396k fetal cells). Same lab, same 10x v2 chemistry as Bhaduri 2020 organoids.

Session 17 downloaded Bhaduri 2021 and saved `bhaduri_2021_raw.h5ad` (396,186 × 33,694) on Drive.

**This notebook builds a balanced 1:1 input for integration:** stratified subsample both datasets to 100k cells each. Stratum floor = 200 cells (hard-drop, no upsampling — upsampling duplicates cells at zero distance in the kNN graph, which is exactly the pathology that broke DPT). Per-stratum targets = proportional to cleaned stratum size (preserves natural composition).

Stratification axes:
- **Bhaduri 2020** (organoids): `protocol × age_week` (organoids have no anatomical area).
- **Bhaduri 2021** (fetal): `age_gw × cell_type_coarse`. `area_ucsc` deliberately left out — it would only apply to the fetal side (asymmetric), fragment strata from ca. 120 → ca. 1,200 (most below the 200 floor), and areal identity is a Phase-2 question.

## Section 0 — Setup

Install `scanpy` (fresh Colab kernel) and mount Drive for persistent h5ad storage.

In [ ]:
!pip install -q scanpy
from google.colab import drive
drive.mount('/content/drive')

## Section 1 — Load Bhaduri 2020 and inspect metadata

Starting file is `bhaduri_2020_clustered.h5ad` — contains raw counts in `adata.raw`, Leiden labels, and sample prefixes on barcodes. Integration-stage transforms (HVG selection, PCA, Harmony) will be redone downstream on the balanced input, so we start from the clustered h5ad rather than the dense-scaled preprocessed one (which was ca. 32 GB anyway and has been deleted).

Key question for subsampling: **does the `obs` already contain `protocol` and `age` columns?** If not, we must derive them from the `sample` prefix.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_clustered.h5ad')
print("Shape:", adata.shape)
print("\nobs columns:", list(adata.obs.columns))
print("\nFirst 5 obs rows:")
print(adata.obs.head())
print("\nUnique samples (first 10):", adata.obs['sample'].unique()[:10] if 'sample' in adata.obs.columns else "NO 'sample' COLUMN")
print("\nSample count:", adata.obs['sample'].nunique() if 'sample' in adata.obs.columns else "N/A")

### Finding

- Shape: **241,776 × 16,774**.
- `obs` columns: `n_genes_by_counts`, `total_counts`, `total_counts_mt`, `pct_counts_mt`, `n_genes`, `sample`, `leiden`. **No `protocol`, no `age`** — must be derived.
- 37 unique samples. Names like `H1SWeek3`, `H28126SWeek5`, `YH10PWeek10` — clearly encode **`{cell_line}{protocol_letter}Week{N}`** where protocol ∈ {S, X, P} (matches the paper's 3-protocol design).

## Section 2 — First parsing attempt

Hypothesis: every sample name follows the clean pattern `^{line}{[SXP]}Week{\d+}$`. One regex should cover all 37.

In [ ]:
import re
pattern = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])Week(?P<age>\d+)$')

all_samples = adata.obs['sample'].unique().tolist()
parsed, failed = [], []
for s in all_samples:
    m = pattern.match(s)
    if m:
        parsed.append((s, m.group('line'), m.group('protocol'), int(m.group('age'))))
    else:
        failed.append(s)

print(f"Parsed: {len(parsed)}/{len(all_samples)}")
print(f"Failed: {failed if failed else 'none'}")

import pandas as pd
df = pd.DataFrame(parsed, columns=['sample', 'line', 'protocol', 'age_week'])
print("\nProtocol counts (samples):", df['protocol'].value_counts().to_dict())
print("Age counts (samples):",       df['age_week'].value_counts().sort_index().to_dict())
print("Line counts (samples):",      df['line'].value_counts().to_dict())
print("\nProtocol × age grid (samples):")
print(df.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))

### Finding

**28/37 parse cleanly. 9 fail.** Two pattern families escape the regex:

1. **`H28126S2Week{5,8,10}`** — cell line `H28126` with `S2` designation (likely protocol S, variant/batch 2). The trailing digit breaks the `{protocol_letter}Week` pattern.
2. **`Week{5,8,10}{S,P}`** — reverse-order naming with no cell-line prefix.

Nine samples sounds small, but we don't know yet how many *cells* they represent — need to check before deciding how careful to be.

## Section 3 — Investigate failed samples

If the 9 unmatched samples carry only a handful of cells, we could drop them. If they carry tens of thousands, we must parse them properly.

In [ ]:
failed_samples = ['H28126S2Week8', 'H28126S2Week5', 'Week5S', 'Week8S',
                  'Week8P', 'Week5P', 'H28126S2Week10', 'Week10S', 'Week10P']
print("Cells per unmatched sample:")
print(adata.obs['sample'].value_counts().loc[failed_samples])
print(f"\nTotal unmatched cells: {adata.obs['sample'].isin(failed_samples).sum()}")
print(f"Of grand total: {adata.obs['sample'].isin(failed_samples).sum() / len(adata) * 100:.1f}%")

### Finding

**57,718 cells (23.9% of the dataset)** are in the 9 unmatched samples. Too large to drop. We must extend the parser.

### Decisions on the two edge-case families

**`H28126S2Week*` (3 samples, 16,053 cells) — fold `S2` into `S`.**
- Our stratification axis is *protocol* for balancing against fetal maturation, not batch-effect analysis.
- Folding matches the paper's stated 3-protocol design.
- Alternative (keep S2 separate) would fragment strata and add a 4th protocol level that isn't biologically distinct.

**`Week{N}{S,P}` (6 samples, 41,665 cells) — label cell line as `unknown`; parse protocol + age normally.**
- Cell line is not a stratification axis, so its absence is acceptable.
- Protocol + age are what we actually need.

## Section 4 — Final parser

Three patterns, tried in order of specificity:

1. `pat_variant` — `{line}{[SXP]}\d+Week{N}` — catches `H28126S2Week*`.
2. `pat_standard` — `{line}{[SXP]}Week{N}` — original clean pattern.
3. `pat_reverse` — `Week{N}{[SXP]}` — catches `Week{N}{S,P}`; line labeled `unknown`.

We then map `(protocol, age_week)` onto every cell and compute the **cell-level grid**, which is what matters for stratification sizing.

In [ ]:
import re

pat_standard = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])Week(?P<age>\d+)$')
pat_variant  = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])\d+Week(?P<age>\d+)$')
pat_reverse  = re.compile(r'^Week(?P<age>\d+)(?P<protocol>[SXP])$')

def parse_sample(s):
    for pat in (pat_variant, pat_standard):  # try variant first (more specific)
        m = pat.match(s)
        if m:
            return m.group('line'), m.group('protocol'), int(m.group('age'))
    m = pat_reverse.match(s)
    if m:
        return 'unknown', m.group('protocol'), int(m.group('age'))
    return None

all_samples = adata.obs['sample'].unique().tolist()
parsed, failed = [], []
for s in all_samples:
    r = parse_sample(s)
    if r:
        parsed.append((s, *r))
    else:
        failed.append(s)

import pandas as pd
df = pd.DataFrame(parsed, columns=['sample', 'line', 'protocol', 'age_week'])

print(f"Parsed: {len(parsed)}/{len(all_samples)}")
print(f"Failed: {failed if failed else 'none'}")
print("\nProtocol × age grid (SAMPLES):")
print(df.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))

# Build two separate mappings (tuple-map on a Categorical series triggers a pandas MultiIndex error)
protocol_map = dict(zip(df['sample'], df['protocol']))
age_map      = dict(zip(df['sample'], df['age_week']))

sample_str = adata.obs['sample'].astype(str)
obs_tmp = pd.DataFrame({
    'protocol': sample_str.map(protocol_map),
    'age_week': sample_str.map(age_map),
})

print("\nProtocol × age grid (CELLS):")
print(obs_tmp.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))
print(f"\nTotal cells with (protocol, age): {obs_tmp.dropna().shape[0]} / {len(obs_tmp)}")

### Finding

**37/37 samples parse. 241,776 / 241,776 cells mapped.**

**Cell-level grid:**

| protocol \ age_week | 3 | 5 | 8 | 10 | 15 | 24 |
|---|---|---|---|---|---|---|
| P | 0 | 14,228 | 16,697 | 13,367 | 0 | 1,591 |
| S | 18,017 | 45,757 | 29,567 | 48,183 | 2,718 | 2,914 |
| X | 20,321 | 10,126 | 2,427 | 15,863 | 0 | 0 |

**Stratum health:** 14 populated strata out of 18 possible (4 empty: P×3, P×15, X×15, X×24). Lowest populated stratum = X×8 at **2,427 cells** — an order of magnitude above the 200-cell floor. **The hard-drop rule removes 0 cells from Bhaduri 2020.**

**Rough per-protocol share at 100k target (proportional):** S ≈ 63k, P ≈ 19k, X ≈ 19k. Reflects the true cohort composition — S was run in more cell lines across more timepoints.

## Section 5 — Bhaduri 2020: stratified subsample to 100k

With `protocol` and `age_week` derived in Section 4, we can run the subsample.

**Method:** proportional per-stratum targets via the largest-remainder rule (guarantees targets sum to exactly 100,000). Hard-drop any stratum with fewer than 200 cells — we already verified none of the 14 populated strata fall below this floor, but the rule is in the code for correctness (identical logic runs on Bhaduri 2021 in Section 6, where strata *do* drop). Fixed seed = 42.

First, write `protocol` and `age_week` back into `adata.obs` as proper columns (Section 4 built them in a temp DataFrame). Then compute per-stratum targets and display them before committing to the subsample — a visual check that numbers sum to 100,000 and match the expected S ≈ 63k, P ≈ 19k, X ≈ 19k split.

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
TARGET = 100_000
FLOOR = 200

# Write protocol/age_week into adata.obs as proper columns
adata.obs['protocol'] = sample_str.map(protocol_map).astype('category')
adata.obs['age_week'] = sample_str.map(age_map).astype('Int64')

# Stratum sizes (drop empty combinations)
strata = adata.obs.groupby(['protocol', 'age_week'], observed=True).size()
strata = strata[strata > 0]

# Apply hard-drop floor
passed = strata[strata >= FLOOR]
dropped = strata[strata < FLOOR]
print(f"Strata: {len(strata)} total, {len(passed)} passed (>={FLOOR}), {len(dropped)} dropped")
if len(dropped):
    print("Dropped:")
    print(dropped)
print(f"Cells available for sampling: {passed.sum()}")

# Largest-remainder proportional targets (guarantees sum = TARGET)
raw = passed * TARGET / passed.sum()
floor_t = np.floor(raw).astype(int)
remainder = TARGET - floor_t.sum()
frac = raw - floor_t
order = np.argsort(-frac.values)
targets = floor_t.copy()
targets.iloc[order[:remainder]] += 1

# Safety cap: target can't exceed stratum size
targets = pd.concat([targets.rename('target'), passed.rename('available')], axis=1)
targets['target'] = targets[['target', 'available']].min(axis=1)

print(f"\nPer-stratum targets (sum = {targets['target'].sum()}):")
print(targets)

# Per-protocol totals (quick sanity)
print("\nPer-protocol totals:")
print(targets.groupby(level='protocol')['target'].sum())

### Finding

Expected: **14 strata passed, 0 dropped** (lowest populated stratum X×8 = 2,427 ≫ 200). Per-stratum targets sum to exactly 100,000. Per-protocol totals roughly **S ≈ 63k, P ≈ 19k, X ≈ 19k** — matching the cell-level grid from Section 4.

If these numbers look right, proceed to the next cell to execute the subsample and save.

### Execute subsample and save

One numpy RNG with fixed seed, one `rng.choice(..., replace=False)` call per passed stratum. Slicing `adata[keep_indices]` preserves `adata.raw` — raw counts are needed for downstream integration (colab_03 re-run on balanced input).

In [ ]:
rng = np.random.default_rng(SEED)
keep_indices = []
for (p, a), row in targets.iterrows():
    mask = (adata.obs['protocol'] == p) & (adata.obs['age_week'] == a)
    stratum_idx = np.where(mask.values)[0]
    chosen = rng.choice(stratum_idx, size=int(row['target']), replace=False)
    keep_indices.extend(chosen.tolist())
keep_indices = sorted(keep_indices)

adata_sub = adata[keep_indices].copy()
print(f"Subsample shape: {adata_sub.shape}")
print(f"adata_sub.raw present: {adata_sub.raw is not None}")

# Composition sanity — proportions preserved?
print("\nProtocol composition (original vs subsample):")
print(pd.DataFrame({
    'original': adata.obs['protocol'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata_sub.obs['protocol'].value_counts(normalize=True).sort_index().round(4),
}))
print("\nAge_week composition (original vs subsample):")
print(pd.DataFrame({
    'original': adata.obs['age_week'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata_sub.obs['age_week'].value_counts(normalize=True).sort_index().round(4),
}))

# Save
OUT_PATH = '/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad'
adata_sub.write_h5ad(OUT_PATH)
print(f"\nSaved: {OUT_PATH}")

# Free memory before Section 6
del adata, adata_sub
import gc; gc.collect()

### Finding

Expected shape: **(100000, 16774)**. Protocol and age_week proportions in the subsample should match the original to within fractions of a percent — confirmation that proportional sampling preserves natural composition.

`adata_sub.raw present` should be `True` (preserved by slicing), so raw counts survive into the saved h5ad. Integration (colab_03 re-run) will start from these raw counts.

File written to Drive at `data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad`. `adata` and `adata_sub` are freed before loading Bhaduri 2021 — Colab memory gets tight with 396k-cell loads.

## Section 6 — Bhaduri 2021: load and apply pre-subsample filters

Load `bhaduri_2021_raw.h5ad` (396,186 × 33,694, built in colab_06 from 74 NeMO samples). Before stratifying we drop two cell populations flagged in the UCSC metadata:

- **`cell_type_coarse == "Outlier"`** — ca. 30k cells. UCSC's explicit quality-control flag; the atlas team excluded these from downstream analysis.
- **`cell_type == "0"`** — ca. 50k cells where the fine-grained consensus classifier could not assign an identity (literal string `"0"`).

Dropping these *before* stratification matters: if we included them, we'd either waste subsample slots on ambiguous cells or have to post-filter (which would break the exact 100,000 target).

After filtering, preview the `age_gw × cell_type_coarse` grid and apply the 200-cell floor. Unlike Bhaduri 2020 (14 populated strata, lowest at 2,427), this grid spans ca. 12 ages × 9 remaining coarse types = up to ca. 108 strata — so rare combinations (e.g. `GW14 × CR`, or late-stage vascular cells) will fall below the floor and drop.

In [ ]:
import scanpy as sc

adata21 = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_raw.h5ad')
print("Shape:", adata21.shape)
print("\nobs columns:", list(adata21.obs.columns))
print("\nFirst 5 obs rows:")
print(adata21.obs.head())
print("\ncell_type_coarse distribution:")
print(adata21.obs['cell_type_coarse'].value_counts())
print("\ncell_type distribution (top 15):")
print(adata21.obs['cell_type'].value_counts().head(15))

### Finding

Confirm `cell_type_coarse`, `cell_type`, and `age_gw` are present as obs columns. Expected `cell_type_coarse` distribution (from Session 17 notes): Neuron ca. 50.8%, Interneuron ca. 15.1%, RG ca. 9.8%, Dividing ca. 8.1%, **Outlier ca. 7.6%** (will drop), IPC ca. 3.4%, Microglia ca. 2.5%, OPC ca. 1.8%, Vascular ca. 0.8%, CR ca. 0.07%.

`cell_type == "0"` should appear in the top-15 list as ca. 50k cells (ca. 13% of dataset) — UCSC's unassigned placeholder.

In [ ]:
n0 = adata21.n_obs
mask = (adata21.obs['cell_type_coarse'] != 'Outlier') & (adata21.obs['cell_type'] != '0')
adata21 = adata21[mask].copy()

dropped = n0 - adata21.n_obs
print(f"Dropped: {dropped:,} cells ({100 * dropped / n0:.1f}%)")
print(f"After filter: {adata21.shape}")

### Finding

Expected ca. 80,000 cells dropped (ca. 20% of 396k): ca. 30k Outlier + ca. 50k `"0"`. Remaining ≈ **316k cells × 33,694 genes** for stratification.

In [ ]:
import pandas as pd
FLOOR = 200

# Cell-level grid: age_gw x cell_type_coarse
grid = adata21.obs.groupby(['age_gw', 'cell_type_coarse'], observed=True).size().unstack(fill_value=0)
print("Cell-level grid (age_gw x cell_type_coarse):")
print(grid)

# Floor check
strata21 = adata21.obs.groupby(['age_gw', 'cell_type_coarse'], observed=True).size()
strata21 = strata21[strata21 > 0]
passed21 = strata21[strata21 >= FLOOR]
dropped21 = strata21[strata21 < FLOOR]
print(f"\nStrata: {len(strata21)} total, {len(passed21)} passed (>={FLOOR}), {len(dropped21)} dropped")
if len(dropped21):
    print(f"\nDropped strata (totaling {dropped21.sum()} cells):")
    print(dropped21.sort_values())
print(f"\nCells in kept strata: {passed21.sum():,} / {strata21.sum():,} ({100 * passed21.sum() / strata21.sum():.1f}%)")

### Finding

The strata that drop here are where the 200-cell floor actually bites — likely rare combinations (e.g. `GW14 × CR`, `GW25 × Vascular`). As long as the dropped-cell fraction is small (<5%), the hard-drop rule is doing its job: removing strata too thin to contribute stable DPT signal, not gutting the dataset.

If a critical cell-type × age combination is dropped, we'd revisit the floor. But those fringe combinations are also the ones most likely to produce DPT artifacts (disconnected components, isolated nodes) — exactly the failure mode Session 15 diagnosed. Dropping them is the right trade.

## Section 7 — Bhaduri 2021: stratified subsample to 100k

Mirror of Section 5 applied to the filtered Bhaduri 2021 (`passed21` from Section 6). Same method: largest-remainder proportional targets, hard-drop floor already applied in Section 6, fixed seed = 42.

The sampling is independent from Bhaduri 2020, but the seed is the same — this just means both draws are deterministic, not that they're correlated (they operate on disjoint data).

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
TARGET = 100_000

# passed21 defined in Section 6; strata already floor-filtered
raw = passed21 * TARGET / passed21.sum()
floor_t = np.floor(raw).astype(int)
remainder = TARGET - floor_t.sum()
frac = raw - floor_t
order = np.argsort(-frac.values)
targets21 = floor_t.copy()
targets21.iloc[order[:remainder]] += 1

# Safety cap: target can't exceed stratum size
targets21 = pd.concat([targets21.rename('target'), passed21.rename('available')], axis=1)
targets21['target'] = targets21[['target', 'available']].min(axis=1)

print(f"Per-stratum targets (sum = {targets21['target'].sum()}):")
print(targets21)
print(f"\nPer-age_gw totals:")
print(targets21.groupby(level='age_gw')['target'].sum())
print(f"\nPer-cell_type_coarse totals:")
print(targets21.groupby(level='cell_type_coarse')['target'].sum())

### Finding

Targets sum to exactly 100,000. Per-`age_gw` totals roughly reflect the fetal developmental spread (GW14–25), weighted by how many cells each timepoint contributed to the filtered set. Per-`cell_type_coarse` totals follow the filtered composition minus any strata that dropped to the floor — Neuron dominates (ca. 50k range), followed by Interneuron (ca. 15k), RG, Dividing, IPC, Microglia, OPC, Vascular, CR.

If a cell type is absent from the targets table, all its age strata fell below the 200-cell floor — most likely candidate is CR (0.07% of dataset).

### Execute subsample and save

Same pattern as Section 5: one numpy RNG seeded at 42, one `rng.choice(..., replace=False)` per passed stratum. Save to `bhaduri_2021_100k.h5ad`, then free memory before Section 8.

In [ ]:
rng = np.random.default_rng(SEED)
keep21 = []
for (gw, ct), row in targets21.iterrows():
    mask = (adata21.obs['age_gw'] == gw) & (adata21.obs['cell_type_coarse'] == ct)
    stratum_idx = np.where(mask.values)[0]
    chosen = rng.choice(stratum_idx, size=int(row['target']), replace=False)
    keep21.extend(chosen.tolist())
keep21 = sorted(keep21)

adata21_sub = adata21[keep21].copy()
print(f"Subsample shape: {adata21_sub.shape}")
print(f"adata21_sub.raw present: {adata21_sub.raw is not None}")

# Composition sanity (filtered original vs subsample)
print("\nage_gw composition (filtered original vs subsample):")
print(pd.DataFrame({
    'original': adata21.obs['age_gw'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata21_sub.obs['age_gw'].value_counts(normalize=True).sort_index().round(4),
}))
print("\ncell_type_coarse composition (filtered original vs subsample):")
print(pd.DataFrame({
    'original': adata21.obs['cell_type_coarse'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata21_sub.obs['cell_type_coarse'].value_counts(normalize=True).sort_index().round(4),
}))

# Save
OUT21 = '/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad'
adata21_sub.write_h5ad(OUT21)
print(f"\nSaved: {OUT21}")

# Free memory before Section 8
del adata21, adata21_sub
import gc; gc.collect()

### Finding

Expected shape: **(100000, 33694)**. `age_gw` and `cell_type_coarse` proportions in the subsample should match the filtered original — except for strata that hard-dropped in Section 6. If a cell type's `original` proportion is nonzero but `subsample` shows 0 or NaN, all that cell type's strata fell below the floor (most likely candidate: CR, at 0.07% of the dataset).

File saved to Drive. `adata21` and `adata21_sub` freed before Section 8 reloads both 100k files for joint checks.

## Section 8 — Joint sanity checks

Load both 100k files back from Drive and verify:

- **Shape parity** — both should be `(100_000, N_genes)`.
- **Gene-space overlap** — the intersection is what `colab_03` will use for integration. Bhaduri 2020 has 16,774 genes; Bhaduri 2021 has 33,694. Expect the intersection to be close to the smaller set.
- **obs column inventory** — confirm stratification axes survived the save round-trip. Columns don't need to match between datasets; `colab_03` will add a `dataset` label during concatenation.

In [ ]:
import scanpy as sc

adata20 = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad')
adata21 = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad')

print("Bhaduri 2020 (organoids):", adata20.shape)
print("  obs columns:", list(adata20.obs.columns))
print("  raw present:", adata20.raw is not None)
print()
print("Bhaduri 2021 (fetal):", adata21.shape)
print("  obs columns:", list(adata21.obs.columns))
print("  raw present:", adata21.raw is not None)

# Gene-space overlap
genes20 = set(adata20.var_names)
genes21 = set(adata21.var_names)
shared = genes20 & genes21
print(f"\nGene space:")
print(f"  Bhaduri 2020: {len(genes20):,} genes")
print(f"  Bhaduri 2021: {len(genes21):,} genes")
print(f"  Shared: {len(shared):,}")
print(f"  Bhaduri 2020 only: {len(genes20 - genes21):,}")
print(f"  Bhaduri 2021 only: {len(genes21 - genes20):,}")

### Finding

Expected:
- Both files at (100000, N_genes).
- Bhaduri 2020: `protocol`, `age_week`, `sample`, `leiden` in obs; `raw present = True`.
- Bhaduri 2021: `age_gw`, `cell_type_coarse`, `cell_type`, `area_ucsc`, `individual`, `donor`, etc. `raw present` depends on how colab_06 saved it.
- Shared gene space: ca. 14,000–16,000 genes — close to Bhaduri 2020's 16,774 minus nomenclature mismatches. This is the integration-ready gene set for colab_03.

If shared gene count drops below ca. 12,000, investigate — likely nomenclature mismatch between the two reference annotations, needs fixing before colab_03.

## Notebook complete

Deliverables:
- `data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad`
- `data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad`

**Next notebook:** re-run `colab_03_integration.ipynb` on these balanced inputs — concatenate on shared gene space, add `dataset` label, normalize, HVG, PCA, Harmony, Leiden, save `integrated_harmony.h5ad`. Then re-run `colab_04` (annotation) and `colab_05` (trajectory) on the balanced integration. The 100×-imbalance failure mode that broke DPT in Session 15 should be gone.